In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from arch.nvae.nvae import NVAE
from const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

model_path = "logs/nvae_acdc/sandbox/checkpoints/epoch=93-step=20116.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = NVAE.load_from_checkpoint(model_path)

In [ ]:
loader_test = data_module.test_dataloader()
x = next(iter(loader_test))
x.shape

In [ ]:
from utils.eval import get_samples_and_reconstructions
from utils.utils import show_samples

with torch.no_grad():
    model.eval()
    model.to(device)
    
    x_hat_logits, _, _, _, _ = model(x)

recon_loss = model.reconstruction_loss(x, x_hat_logits)
print(recon_loss)

samples_and_reconstructions = get_samples_and_reconstructions(x[:20], x_hat_logits[:20])
show_samples(samples_and_reconstructions, rgb=False, ncol=10, figsize=(10, 4), display=False)